# RAG Evaluation — DiabetesAssist
Tests the RAG pipeline with diabetes questions and scores answers using LLM-as-Judge.

 We import libraries used specifically for evaluation tasks.
 These libraries are only required for running the evaluation pipeline and metrics.

In [184]:
%%capture
! pip install -r  requirements_eval.txt

In [1]:
import sys
sys.path.append('..')  # so we can import from the project root
#load env variables
from dotenv import load_dotenv
load_dotenv('../.env')

import os
import json

from langfuse import Langfuse
import anthropic

from IPython.display import HTML

import requests


### Langfuse + Anthropic Pipeline
We initialize the Langfuse and Anthropic clients. Langfuse is used for observability, tracing, and evaluation logging, while the Anthropic client is used to access Claude models through the API. 
Note You must start you server this is a toy example:

In [2]:

lf = Langfuse(
    public_key=os.environ["LANGFUSE_PUBLIC_KEY"],
    secret_key=os.environ["LANGFUSE_SECRET_KEY"],
    host=os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com")
)
client = anthropic.Anthropic()


## Correctness Evaluation

Correctness measures whether the chatbot's answer is **factually accurate** compared to a known correct answer.



### How it works

You need three things:

- **Question** — what the user asked
- **Correct answer** — the ground truth, written by a human or extracted from a trusted document
- **Chatbot answer** — what the chatbot actually said

The judge (Claude) compares the chatbot answer against the correct answer and returns a score.



### Scoring

| Score | Meaning |
|-------|---------|
| 1 | Completely wrong |
| 2 | Mostly wrong |
| 3 | Partially correct |
| 4 | Mostly correct |
| 5 | Fully correct |



In [3]:
question = "What is HbA1c?"

correct_answer = "HbA1c measures average blood sugar over 2-3 months. Diabetics aim below 7%."

chatbot_answer = "HbA1c is a blood test showing average blood sugar over 3 months."

This is a hardcoded toy example used to verify the judge pipeline works before connecting it to real Langfuse traces.





In [4]:

def evaluate_correctness( question, correct_answer, chatbot_answer):
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=100,
        messages=[{"role": "user", "content": f"""Score this answer 1-5 for correctness against the correct answer.

Question: {question}
Correct answer: {correct_answer}
Chatbot answer: {chatbot_answer}

Reply with a single number only."""}]
    )
    
    score = int(response.content[0].text.strip())
    return score
   


we can evaluate the chatbot answer like this by calling the function:

In [5]:
evaluate_correctness( question, correct_answer, chatbot_answer)

4

## Agent Correctness Evaluation with LangSmith

We want to test the correctness of our agent using [LangSmith](https://smith.langchain.com/). We will use the dataset that we have already uploaded to LangSmith to evaluate the agent’s performance and validate its outputs across different test cases.

we can load the dataset into LangSmith

In [6]:
dataset = lf.get_dataset("medical-rag-eval")
dataset 

dataset is currently a DatasetClient object, not the fully loaded dataset contents.
This is typically a lazy-loaded client/iterator object that provides access to the dataset without immediately fetching all items into memory. Was can get the items by using the items method, we can get the expected input and ouput

In [7]:

input =  dataset.items[0].input['text']['question']
expected_output =  dataset.items[0].expected_output['text']['answer']

print( "Input: ", input )
print( "Expected Output: ", expected_output )

Input:  What condition is also known as long COVID?
Expected Output:  Post COVID-19 condition (PCC).


Langfuse is hosted in the cloud (handles dataset storage and tracing) but we call the agent directly via HTTP POST — no UI needed. Change the URL below depending on where your agent is running:

Local:  http://localhost:8000

EC2:    http://<your-ec2-ip>:8000/


In [8]:
url="http://localhost:8000"

agent_endpoint = f"{url}/chat"
#clear previous conversations for speed and consistency in evaluation
requests.post(f"{url}/reset")

response = requests.post(
    agent_endpoint,
    json={"message": "What is the HbA1c target for type 2 diabetes?"},

)


In [9]:
print("status_code:", response.status_code)   # 200
print("headers:", response.headers)       # content-type, etc.
print("text:", response.text)          # the answer string
print("elapsed:", response.elapsed)       # how long it took
  

status_code: 200
headers: {'date': 'Sat, 09 May 2026 02:13:48 GMT', 'server': 'uvicorn', 'content-type': 'text/plain; charset=utf-8', 'transfer-encoding': 'chunked'}
text: Short answer
- HbA1c targets are individualized. For many nonpregnant adults with type 2 diabetes the American Diabetes Association recommends a general target of <7.0% (individualize up or down based on patient factors). (American Diabetes Association, Standards of Care in Diabetes—2026.)  

What that means in practice
- A lower (more stringent) target (for example, around or below 6.5%) may be appropriate for some younger, otherwise healthy people if it can be achieved without hypoglycemia or undue treatment burden.  
- A less stringent target (for example, higher than 7.0%, sometimes up to ~8.0% or more) may be appropriate for people with limited life expectancy, multiple comorbidities, long diabetes duration, or high hypoglycemia risk.  
These principles and the need to individualize goals are described in the AD

we can also evaluate correctness here by calling the function directly inputting the question, expected output, and chatbot answer 

In [10]:

chatbot_answer = response.text
correct_score=evaluate_correctness( question, expected_output, chatbot_answer)


In [11]:

print("question",question)
print("Chatbot answer:", chatbot_answer)
print("Expected output:", expected_output)
print("Correctness score:", correct_score)

question What is HbA1c?
Chatbot answer: Short answer
- HbA1c targets are individualized. For many nonpregnant adults with type 2 diabetes the American Diabetes Association recommends a general target of <7.0% (individualize up or down based on patient factors). (American Diabetes Association, Standards of Care in Diabetes—2026.)  

What that means in practice
- A lower (more stringent) target (for example, around or below 6.5%) may be appropriate for some younger, otherwise healthy people if it can be achieved without hypoglycemia or undue treatment burden.  
- A less stringent target (for example, higher than 7.0%, sometimes up to ~8.0% or more) may be appropriate for people with limited life expectancy, multiple comorbidities, long diabetes duration, or high hypoglycemia risk.  
These principles and the need to individualize goals are described in the ADA Standards of Care. (American Diabetes Association, Standards of Care in Diabetes—2026.)

Not medical advice
- This is general guid

This function logs a single evaluation run to Langfuse. It opens an observation called "rag-eval" and records the question as the input and both the chatbot answer and the raw LLM response as the output. While inside that observation it grabs the auto-generated trace_id. Once the observation closes, it attaches a correctness score to that trace using the trace_id, flushes the data to Langfuse cloud, and returns the trace_id in case you need it later.

In [12]:


def log_eval(question, answer, llm_response, correct_score, name="correctness"):
    with lf.start_as_current_observation(name="rag-eval") as obs:
        obs.update(
            input={"question": question},
            output={"answer": answer, "llm_response": llm_response},
        )
        trace_id = lf.get_current_trace_id()

    lf.create_score(trace_id=trace_id, name=name, value=correct_score)
    lf.flush()
    return trace_id

we can log the score we see the trace id

In [13]:
log_eval(question, expected_output, chatbot_answer, correct_score)

'53d88db95865e24bcf4523c3db357c3a'

You can view the score on Langfuse UI :![alt text](img/langfuse_score.png)

we can load the dataset 

let Loop through every item in the Langfuse dataset and calcuate the correctenss 

In [14]:
# Loop through every item in the Langfuse dataset
for item in dataset.items:
    
    # Extract the question and expected answer from the dataset item
    question = item.input['text']['question']
    expected_output = item.expected_output['text']['answer']

    # Reset the RAG chatbot's conversation history for a clean slate
    requests.post(f"{url}/reset")
    
    # Send the question to the RAG chatbot running locally

    response = requests.post(
        agent_endpoint,
        json={"message": question},
    )
    
    # Get the chatbot's raw response text
    chatbot_answer = response.text
    
    # Ask Claude Haiku to score the chatbot answer 1-5 for correctness
    # comparing it against the expected answer
    correct_score = evaluate_correctness(question, expected_output, chatbot_answer)
    
    # Log the question, answers, and score to Langfuse
    log_eval(question, expected_output, chatbot_answer, correct_score)




dataset

## Langfuse: Using LLM as Judge

Langfuse allows you to create an LLM judge directly from the UI — no code required. This is useful in a team setting where non-engineers (such as a clinician) can configure or adjust evaluations without touching the codebase.

There are built-in templates. Some are straightforward — relevance only needs the input and output. Others like faithfulness require additional instrumentation, specifically logging the retrieval span from the tool. This notebook covers both, but note that faithfulness requires a small change to `tool.py`.



### Relevance

Relevance measures whether the answer actually addresses what the user asked. A score of 1 means the answer directly responds to the question; a score of 0 means it is off-topic or misses the point entirely.

**Example:** A user asks *"What is the starting dose of metformin?"* and the model responds with a full explanation of metformin dosing — relevance score of 1. If the model instead returns general information about type 2 diabetes without mentioning dosing, the relevance score would be low.


### Faithfulness

Faithfulness measures whether the model's answer is grounded in the retrieved chunks. If the answer contains claims not supported by the context, the model is hallucinating.

You need three things to score it:

- **question** — what the user asked
- **context** — the chunks retrieved from Chroma
- **answer** — what the model said

**Why you can't score it with a POST request alone:**

When you call `/chat`, the endpoint returns a streamed text response — just the answer. The retrieved chunks are internal to the agent and never returned to the caller. To score faithfulness you need the chunks, but they are consumed inside `document_search` and discarded before the response is sent.

This is why we log the retrieval span to Langfuse — it captures the chunks at the point they are retrieved, inside the trace, without changing the API.



### Setting Up in Langfuse

1. Go to **Evaluations** → **Create Evaluator**
2. Select **LLM-as-a-judge**
3. Pick **Relevance** from the Langfuse maintained templates — faithfulness requires a custom prompt and additional instrumentation
4. Select **Claude** as the judge model
5. You dont have to Map the variables is this case as we only need the input and output


In [20]:
from IPython.display import HTML

HTML("""
<video width="800" controls>
  <source src="https://static.langfuse.com/prod-assets/onboarding/scores-llm-as-a-judge-overview-v1.mp4" type="video/mp4">
</video>
""")

### Langfuse Evaluation Prompts

Langfuse uses LLM-as-a-judge to score traces automatically. The judge is given a prompt that defines the scoring criteria and examples. Below are the prompts used in this project.



### Relevance (0 to 1)

Langfuse maintained template. Scores how well the answer addresses the user's question, staying focused without straying into unrelated areas.

**Prompt:**

> Evaluate the relevance of the generation on a continuous scale from 0 to 1. A generation can be considered relevant (Score: 1) if it enhances or clarifies the response, adding value to the user's comprehension of the topic in question. Relevance is determined by the extent to which the provided information addresses the specific question asked, staying focused on the subject without straying into unrelated areas or providing extraneous details.
>
> Example:
> Query: Can eating carrots improve your vision?
> Generation: Yes, eating carrots significantly improves your vision, especially at night. This is why people who eat lots of carrots never need glasses. Anyone who tells you otherwise is probably trying to sell you expensive eyewear...
> Score: 0.1
> Reasoning: Only the first part of the first sentence clearly answers the question. The rest is off-topic.

**Variables:**
- `{{query}}` → the user's question
- `{{generation}}` → the model's answer


Once you save it, you can make a call to your agent. You can then select and initialize it.

![LLM as Judge](img/llm_as_jugdge.png)

You can then make  a call to the  agent  and see the result for the score.

![Score Dashboard](img/score.png)


## Faithfulness

In faithfulness evaluation, we want to judge how well the model's answer is supported by the retrieved documents.

This can be a little confusing because the model output becomes the `output`, while the retrieved documents are passed as the `context`.

We now map the variables as follows:

5. Map the variables:
   - `input` → trace input (the question)
   - `output` → trace output (the answer)
   - `context` → retrieval span named **`"retrieval"`** — this name must match exactly what is set in `tool.py`

We need to edit `tool.py` to log the retrieved documents as a span with the name `"retrieval"`. This allows us to use the retrieved documents as context when asking the judge model (Claude Haiku) to evaluate faithfulness.  


```python
# tool.py
with lf.start_as_current_observation(as_type="span", name="retrieval") as obs:
    obs.update(
        input={"query": query, "k": k},
        output=[
            {"content": doc.page_content, "score": float(score)}
            for doc, score in results
        ],
    )
```

Let's set up the  Go to Evaluations → Running Evaluators → Set up evaluator 
Click Create new evaluator




Set Name: Faithfulness

Leave model as default (Claude Sonnet)
Paste your evaluation prompt into the Evaluation prompt box


>You are an expert clinical evaluator assessing whether a medical chatbot's answer is grounded in the provided source documents.
>
>Context (retrieved document chunks):
>{{context}}
>
>Answer (model response):
>{{answer}}
>
>Instructions:
>1. Break the answer into individual factual claims.
>2. For each claim, check if it is explicitly supported by the context.
>3. Assign each claim: 1 if supported, 0 if not supported or contradicted.
>4. Score = sum of 1s / total claims



is should look like this



![new_evel](img/crate_new_eval.png)

Click Save

On the next screen set Run on: Observations
Toggle Run on live incoming observations: on
![new_evel](img/next_observa.png)

Set variable mapping:


{{context}} → Object Field: Output → JsonPath: $.output
{{answer}} → Object Field: Output → JsonPath: $.content


Click Execute



In [ ]:


question = "What is HbA1c and why is it important for diabetics?"
context = """
HbA1c measures average blood sugar over 2-3 months.
Diabetics aim to keep it below 7% to avoid complications.
"""
answer = "HbA1c is a blood test showing average blood sugar over 3 months. Diabetics aim below 7%."
role= "user"
content="Score this answer 1-10 for correctness based on the context."